### Notebook for ETL of indicators for individual works  

#### For some indicators we know the "exogenous" total, for others only the "endogenous" values for the corpus proper

-  number of citations
-  number of references  
-  number of pages  
-  references/page  
-  fwci from oa  
-  fwci from corpus  
-  hc status from oa  
-  hc status from corpus  
-  copied references  
-  copied reference ratio  
-  disruption index  
-  citer authors FD  
-  cited authors FD  
-  citer institutions FD  
-  cited institutions FD  
-  self references  
-  self referencing rate per reference  
-  citer topics FD  
-  cited topics FD  
-  journal spectral rank  
-  institution spectral rank  


In [105]:
%run common_setup.ipynb

#### This cell extracts Works data and transforms it to the work_indicators

- Construct time-series of citations from reference lists  
- Compute the centile for each publication year
- Filter highly cited papers  
- Group the authors of hte highly cited papers

In [ ]:
class CorpusETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def citation_count(self):
        sql = """
            CREATE OR REPLACE TABLE memory.citation_count AS  
            -- ETL FOR citation_count
            -- ======================
            SELECT cited_id AS work_id,
                    cited_by_count,
                    count(citer_id) AS cited_by_count_endogenous
            FROM project.citer_cited
            LEFT JOIN project.raw
            ON id = cited_id
            -- WHERE len(authorships) > 0
            GROUP BY ALL
            ORDER BY cited_by_count_endogenous DESC           
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.citation_count").show()
        return
    
    def references_per_page(self):
        sql = """
            CREATE OR REPLACE TABLE memory.references_per_page AS
            -- ETL FOR references_per_page
            -- ===========================  
            SELECT id AS work_id,
                    referenced_works_count,
                    try_cast("biblio.last_page" AS INT) - try_cast("biblio.first_page" AS INT) AS page_count,
                    referenced_works_count/page_count AS references_per_page
                FROM project.raw
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.references_per_page").show()
        return
    
    def fwci(self):
        sql = """
            CREATE OR REPLACE TABLE memory.fwci AS
            WITH
            get_fwci_endogenous_CTE AS
                (SELECT DISTINCT cited_id,
                                count(citer_id) OVER (PARTITION BY cited_id)/
                                    (count(citer_id) OVER (PARTITION BY cited_year)/
                                    count(DISTINCT citer_id) OVER (PARTITION BY cited_year)) AS fwci_endogenous,
                FROM project.citer_cited
                ORDER BY fwci_endogenous DESC
                )
            SELECT id AS work_id,
                    fwci,
                    fwci_endogenous
                FROM project.raw
                LEFT JOIN get_fwci_endogenous_CTE
                ON id = cited_id
            ORDER BY fwci DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.fwci").show()
        return
    
    def highly_cited(self):
        sql = """
            CREATE OR REPLACE TABLE memory.highly_cited AS  
            -- ETL FOR highly_cited_work
            -- =========================
            WITH
            citation_counts_CTE AS
                (SELECT DISTINCT cited_by_count,
                        count(citer_id) OVER (PARTITION BY cited_id) AS cited_by_count_endogenous,
                        cited_id,
                        cited_year,
                FROM project.citer_cited
                LEFT JOIN project.raw
                ON id = cited_id
                ORDER BY cited_by_count_endogenous DESC
                )

            SELECT cited_id AS work_id,
                    percent_rank(ORDER BY cited_by_count) OVER w AS percent_rank_total,
                    percent_rank(ORDER BY cited_by_count_endogenous) OVER w AS percent_rank_total_endogenous,
            FROM citation_counts_CTE
            WINDOW w AS (PARTITION BY cited_year) -- ORDER BY cited_by_count_total, cited_by_count_endogenous) 
            ORDER BY percent_rank_total DESC, percent_rank_total_endogenous DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.highly_cited").show()
        return
    
    def copied_references(self):
        sql = """
            CREATE OR REPLACE TABLE memory.copied_references AS
            -- ETL FOR copied_references
            -- =========================
            WITH
            get_copied_CTE AS
                (SELECT r1.citer_id,
                        count(r2.cited_id) AS copied_count,
                        referenced_works_count AS total_count,
                    FROM project.citer_cited r1
                    LEFT JOIN project.citer_cited r2
                    ON r1.cited_id = r2.citer_id
                    LEFT JOIN project.raw
                    ON id = r1.citer_id
                    WHERE r1.cited_id = r2.cited_id
                    GROUP BY ALL
                )

            SELECT citer_id AS work_id,
                    total_count,
                    copied_count/total_count AS copied_fraction
            FROM get_copied_CTE
            ORDER BY copied_fraction DESC, total_count DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.copied_references").show()
        return
    
    def disruption_index(self):
        sql = """
            CREATE OR REPLACE TABLE memory.disruption_index AS  
            -- ETL FOR disruption_index
            -- ========================
            WITH
            extract_work_CTE AS
                (SELECT id AS work_id,
                        fwci,
                        unnest(referenced_works) AS referenced_work
                FROM project.raw
                ),
            extract_disruption_index_CTE AS
                (SELECT work_id,
                        r1.fwci/avg(r2.fwci) OVER (PARTITION BY work_id) AS disruption_index
                FROM extract_work_CTE r1
                LEFT JOIN project.raw r2
                ON referenced_work = r2.id
                ORDER BY disruption_index DESC
                )

            SELECT DISTINCT *
            FROM extract_disruption_index_CTE
            WHERE disruption_index NOT NULL AND disruption_index != 'NaN' AND disruption_index != 'Infinity'
            ORDER BY disruption_index DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.disruption_index").show()
        return
    
    def self_references(self):
        sql = """
            CREATE OR REPLACE TABLE memory.self_references AS  
            -- ETL FOR self_references
            -- =======================
            WITH
            get_selfcited_CTE AS
                (SELECT DISTINCT cited_id,
                        isSelfCited
                FROM
                    (SELECT citer_id,
                            cited_id,
                            CASE WHEN list_has_any(citer_authors, cited_authors) = true IS true THEN 1 ELSE 0 END AS isSelfCited
                        FROM (SELECT citer_id, 
                                    list(citer.author_id) AS citer_authors,
                                    cited_id,
                                    list(cited.author_id) AS cited_authors
                            FROM project.citer_cited
                            INNER JOIN project.authorships citer
                            ON citer_id = citer.work_id
                            INNER JOIN project.authorships cited
                            ON cited_id = cited.work_id
                            GROUP BY citer_id, cited_id
                            )
                    )
                )

            SELECT cited_id AS work_id,
                    isSelfCited
            FROM get_selfcited_CTE
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.self_references").show()
        return
    
    def topic_indicators(self):
        sql = """
            CREATE OR REPLACE TABLE memory.topic_indicators AS  
            -- ETL FOR topic_indicators
            -- =======================
            WITH
                topics_CTE AS
                (SELECT work_id,
                        domain_id[-1:] AS domain_id,
                        field_id[-2:] AS field_id,
                        subfield_id[-4:] AS subfield_id
                    FROM project.topics
                ),
                citer_cited_topics_CTE AS
                (SELECT citer_id,
                        t1.domain_id AS domain_id_citer,
                        t1.field_id AS field_id_citer,
                        t1.subfield_id AS subfield_id_citer,
                        cited_id,
                        t2.domain_id AS domain_id_cited,
                        t2.field_id AS field_id_cited,
                        t2.subfield_id AS subfield_id_cited,
                    FROM project.citer_cited
                    LEFT JOIN topics_CTE as t1
                    ON citer_id = t1.work_id
                    LEFT JOIN topics_CTE t2
                    ON cited_id = t2.work_id
                ),
                cited_subfield_list_CTE AS
                (SELECT citer_id,
                        list(subfield_id_cited) AS cited_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY citer_id
                ),
                citer_subfield_list_CTE AS
                (SELECT cited_id,
                        list(subfield_id_citer) AS citer_subfield_list
                    FROM citer_cited_topics_CTE
                    GROUP BY cited_id
                ),
                citer_topic_CTE AS
                (SELECT citer_id,
                        subfield_id_citer,
                        list_distinct(cited_subfield_list) AS cited_topics,
                        len(list_distinct(cited_subfield_list)) AS cited_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN cited_subfield_list_CTE
                    USING (citer_id)
                    GROUP BY ALL
                    ORDER BY cited_topics_count DESC
                ),
                cited_topic_CTE AS
                (SELECT cited_id,
                        subfield_id_cited,
                        list_distinct(citer_subfield_list) AS citer_topics,
                        len(list_distinct(citer_subfield_list)) AS citer_topics_count
                    FROM citer_cited_topics_CTE cct
                    LEFT JOIN citer_subfield_list_CTE
                    USING (cited_id)    cetl.combiner()
                    GROUP BY ALL
                    ORDER BY citer_topics_count DESC
                )

            SELECT citer_id AS work_id,
                    subfield_id_citer AS subfield_id,
                    cited_topics,
                    cited_topics_count,
                    citer_topics,
                    citer_topics_count
            FROM citer_topic_CTE
            LEFT JOIN cited_topic_CTE
            ON citer_id = cited_id
            ORDER BY citer_topics_count DESC
            """
        self.db.sql(sql) #.show()
        self.db.sql("SELECT * FROM memory.topic_indicators").show()
        return
    
    def spectral_ranks(self):
        sql = """  
                -- ETL FOR pagerank AND influence
                -- ==============================
                WITH
                    select_work_journal_CTE AS
                    (SELECT id AS work_id,
                            "primary_location.source".id AS source_id,
                            pagerank,
                            influence,
                        FROM project.raw
                        LEFT JOIN project.pagerank_source
                        ON "primary_location.source".id = citer
                    ),
                    select_work_institutions_CTE AS
                    (SELECT DISTINCT work_id,
                            institution_id,
                            count(DISTINCT institution_id) OVER (PARTITION BY work_id) AS institution_count,
                            pagerank,
                            influence
                        FROM project.authorships
                        LEFT JOIN project.pagerank_institution
                        ON institution_id = citer 
                    ),
                    select_work_institution_CTE AS
                    (SELECT work_id,
                            sum(pagerank)/institution_count AS pagerank_institution,
                            sum(influence)/institution_count AS influence_institution
                        FROM select_work_institutions_CTE
                        GROUP BY work_id, institution_count
                    )

                SELECT work_id,
                        pagerank AS pagerank_source,
                        influence AS influence_source,
                        pagerank_institution,
                        influence_institution
                FROM select_work_journal_CTE
                LEFT JOIN select_work_institution_CTE
                USING (work_id)
                ORDER BY pagerank_institution DESC, pagerank_source DESC
            """
        self.db.sql(sql).show()

    def combiner(self):
        self.db.sql("SHOW ALL TABLES").show()
        sql = "SELECT * FROM memory.citation_count\n"
        for tab in ['references_per_page', 'fwci', 'highly_cited', 'copied_references', 'disruption_index', 'self_references', 'topic_indicators']:
            sql = sql + f"LEFT JOIN (SELECT * FROM memory.{tab}) USING (work_id)\n"
        print(f'{sql = }')
        print(self.db.sql(sql).df().head(16))
        return
            


In [ ]:
def main():

    cetl = CorpusETL()
    cetl.citation_count()
    cetl.references_per_page()
    cetl.fwci()
    cetl.highly_cited()
    cetl.copied_references()
    cetl.disruption_index()
    cetl.self_references()
    cetl.combiner()
    cetl.topic_indicators()
    cetl.combiner()
    # cetl.spectral_ranks()

    # cetl.citation_summary()

    cetl.db.close()
        
    return

In [108]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ backup   │ main    │ authors_full         │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ backup   │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR]  │ false     │
│ backup   │ main    │ citer_cited          │ [citer_id, citer_y…  │ [VARCHAR, BIGINT, VARCHAR, BIGINT, …  │ false     │
│ backup   │ main    │ raw                  │ [id, doi, title, p…  │ [VARCHAR, VARCHAR, VARCHAR, BIGINT,…  │ false     │
│ backup   │ main    │ sources_o